# NBA 3.1 — Feature Table + Time Travel Demo (monthly refresh)

A self-contained example using a small **synthetic** HCP feature DataFrame (5 NPIs, monthly cohorts) in the sandbox schema `nba_fv_test`. No production tables are touched.

## What this notebook shows

| Step | What | Why it matters |
| --- | --- | --- |
| 1 | Build a synthetic monthly feature DataFrame | Stand-in for our real pipeline output |
| 2 | Register it as a **time-series feature table** (UC Delta table + `TIMESERIES` primary key) and set retention | Gives Features UI, point-in-time joins, lineage |
| 3 | Simulate **5 monthly refreshes** (overwrite) | Mirrors the monthly pipeline |
| 4 | **Time travel**: read any previous refresh by version / timestamp | Rollback, audit, reproduce an old model's inputs |
| 5 | **Point-in-time lookup** with `FeatureLookup` | Main feature-table benefit: no future leakage |
| 6 | **Retention**: time travel is *time-based*, not *count-based* | How to keep ~3 past refreshes |
| 7 | **Exact "last 3 refreshes"** with a snapshot table + `REFRESH_DATE` | When you need exactly 3, not ~3 |

## Parquet vs UC table vs feature table

| Capability | Parquet file | Plain UC Delta table | UC feature table (PK + TIMESERIES) |
| --- | --- | --- | --- |
| Governance / permissions / lineage in UC | No (volume-level only) | Yes | Yes |
| Time travel to previous versions | No | Yes | Yes (same Delta mechanism) |
| Features UI, searchable feature catalog | No | No | Yes |
| Point-in-time joins (`create_training_set`) | No | No | Yes |
| Model-to-feature lineage (`fe.log_model`) | No | No | Yes |
| Extra effort | — | — | PK (NOT NULL + unique) + one `create_table()` call |

**Key point:** time travel comes from **Delta Lake**, not from Feature Views. A feature table *is* a Delta table, so it inherits time travel.

In [0]:
%pip install -q databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F

fe = FeatureEngineeringClient()

CAT = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA = "nba_fv_test"                     # sandbox schema (safe to drop)
FT_NAME = f"{CAT}.{SCHEMA}.demo_hcp_features_ft"          # feature table (current version + Delta time travel)
HIST_NAME = f"{CAT}.{SCHEMA}.demo_hcp_features_snapshots" # snapshot table (exactly last N refreshes)
KEEP_N_REFRESHES = 3

def q(name: str) -> str:
    """Backtick-quote a 3-part name (catalog has hyphens)."""
    return ".".join(f"`{p}`" for p in name.split("."))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CAT}`.`{SCHEMA}`")

# Re-runnable demo: start from a clean slate (sandbox demo tables only)
spark.sql(f"DROP TABLE IF EXISTS {q(FT_NAME)}")
spark.sql(f"DROP TABLE IF EXISTS {q(HIST_NAME)}")
print("Ready:", FT_NAME, "|", HIST_NAME)

## Step 1: A synthetic monthly feature DataFrame

Each monthly refresh produces one row per `NPI_NUMBER` per cohort cutoff date (`CUTOFF_DATE_TS`). Every refresh adds one new cohort month and **re-computes** older values (e.g. late-arriving claims), so values differ between refreshes — which is exactly what time travel lets us inspect.

In [0]:
# 5 monthly pipeline runs. Run k contains cohorts Jan-2026 .. month (k+5)
REFRESH_DATES = ["2026-07-05", "2026-08-05", "2026-09-05", "2026-10-05", "2026-11-05"]

def build_refresh_df(refresh_date: str, run_idx: int, include_refresh_date: bool = False):
    n_cohorts = 5 + run_idx           # run 1 -> 6 cohorts (Jan-Jun), run 5 -> 10 cohorts (Jan-Oct)
    seed = 100 + run_idx              # different seed per run -> values restated each refresh
    df = spark.sql(f"""
        SELECT CONCAT('NPI_', LPAD(CAST(n.id AS STRING), 3, '0'))                 AS NPI_NUMBER,
               LAST_DAY(ADD_MONTHS(DATE'2026-01-01', CAST(c.id AS INT)))          AS CUTOFF_DATE_TS,
               CAST(FLOOR(RAND({seed}) * 50) AS BIGINT)                           AS TOTAL_RX_CLAIMS_IN_LAST_1_MONTH,
               CAST(FLOOR(RAND({seed} + 1) * 150) AS BIGINT)                      AS TOTAL_RX_CLAIMS_IN_LAST_3_MONTH,
               CAST(FLOOR(RAND({seed} + 2) * 10) AS BIGINT)                       AS TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH,
               DATE'{refresh_date}'                                               AS REFRESH_DATE
        FROM range(1, 6) AS n CROSS JOIN range(0, {n_cohorts}) AS c
    """)
    return df if include_refresh_date else df.drop("REFRESH_DATE")

display(build_refresh_df(REFRESH_DATES[0], 1).orderBy("NPI_NUMBER", "CUTOFF_DATE_TS").limit(10))

## Step 2: Register as a time-series feature table and set retention

* `create_table()` creates a UC Delta table with `PRIMARY KEY (NPI_NUMBER, CUTOFF_DATE_TS TIMESERIES)`. That is what makes it a feature table (Features UI + point-in-time joins).
* Retention is set **in days**. With a monthly refresh, **~95 days ≈ the last 3 refreshes** (plus a few days of slack for late runs).
  * `delta.deletedFileRetentionDuration` — how long old data files survive `VACUUM` (default 7 days → almost no monthly history!)
  * `delta.logRetentionDuration` — how long version history is kept (default 30 days). Must be ≥ the above.

In [0]:
RETENTION = "interval 95 days"   # ~3 monthly refreshes

fe.create_table(
    name=FT_NAME,
    primary_keys=["NPI_NUMBER", "CUTOFF_DATE_TS"],
    timeseries_column="CUTOFF_DATE_TS",
    schema=build_refresh_df(REFRESH_DATES[0], 1).schema,   # empty table; data written in Step 3
    description="DEMO: synthetic HCP monthly features (time-series feature table)",
)

spark.sql(f"""
ALTER TABLE {q(FT_NAME)} SET TBLPROPERTIES (
  'delta.logRetentionDuration'         = '{RETENTION}',
  'delta.deletedFileRetentionDuration' = '{RETENTION}'
)""")

display(spark.sql(f"SHOW TBLPROPERTIES {q(FT_NAME)}").filter(F.col("key").like("delta.%Retention%")))
ft = fe.get_table(name=FT_NAME)
print("Primary keys:", ft.primary_keys, "| Timeseries key:", ft.timestamp_keys)

## Step 3: Simulate 5 monthly refreshes

Each run **overwrites** the feature table with the newly computed DataFrame (the PK constraint and table properties are kept). We tag every commit with `userMetadata = refresh_date=...` so each Delta version can be traced back to the pipeline run that produced it.

Note: `fe.write_table()` only supports `mode="merge"` (upsert on the primary key). For a full monthly recompute, a Delta overwrite is simpler and avoids leftover stale rows.

In [0]:
def monthly_refresh_feature_table(refresh_date: str, run_idx: int):
    df = build_refresh_df(refresh_date, run_idx)
    (df.write.format("delta")
       .mode("overwrite")
       .option("userMetadata", f"refresh_date={refresh_date}")
       .saveAsTable(q(FT_NAME)))

for i, d in enumerate(REFRESH_DATES, start=1):
    monthly_refresh_feature_table(d, i)
    print(f"Refresh {i} ({d}) written: {spark.table(q(FT_NAME)).count()} rows")

In [0]:
# Sanity check: the monthly overwrite must keep the feature-table PK and the retention properties
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {q(FT_NAME)}").collect()}
ft_after = fe.get_table(name=FT_NAME)
print("Primary keys:", ft_after.primary_keys, "| Timeseries key:", ft_after.timestamp_keys)
print("logRetentionDuration:", props.get("delta.logRetentionDuration"),
      "| deletedFileRetentionDuration:", props.get("delta.deletedFileRetentionDuration"))
assert props.get("delta.deletedFileRetentionDuration") == RETENTION, "Retention property lost after overwrite"
assert "CUTOFF_DATE_TS" in ft_after.timestamp_keys, "TIMESERIES key lost after overwrite"

## Step 4: Time travel to previous refreshes

`DESCRIBE HISTORY` lists every Delta version. Note that **one refresh is not one version**: table creation, property changes, `OPTIMIZE`, etc. also create versions. That's why "keep 3 versions" must be expressed as "keep ~3 refreshes worth of days".

In [0]:
hist = (spark.sql(f"DESCRIBE HISTORY {q(FT_NAME)}")
          .select("version", "timestamp", "operation", "userMetadata",
                  F.col("operationMetrics.numOutputRows").alias("rows_written"))
          .orderBy("version"))
display(hist)

# Map each pipeline refresh to its Delta version
refresh_versions = {r["userMetadata"].split("=")[1]: r["version"]
                    for r in hist.filter(F.col("userMetadata").startswith("refresh_date=")).collect()}
print("refresh_date -> version:", refresh_versions)

In [0]:
v_first  = refresh_versions[REFRESH_DATES[0]]
v_prev   = refresh_versions[REFRESH_DATES[-2]]
v_latest = refresh_versions[REFRESH_DATES[-1]]

# By version
df_first = spark.read.option("versionAsOf", v_first).table(q(FT_NAME))
df_prev  = spark.sql(f"SELECT * FROM {q(FT_NAME)} VERSION AS OF {v_prev}")
df_now   = spark.table(q(FT_NAME))
print(f"Rows  first refresh (v{v_first}): {df_first.count()} | previous (v{v_prev}): {df_prev.count()} | latest: {df_now.count()}")

# By timestamp (e.g. 'what did the table look like right after the previous refresh?')
ts_prev = hist.filter(F.col("version") == v_prev).first()["timestamp"]
df_prev_ts = spark.read.option("timestampAsOf", ts_prev.isoformat()).table(q(FT_NAME))
print(f"TIMESTAMP AS OF {ts_prev}: {df_prev_ts.count()} rows")

# Same NPI / cohort, value as of each refresh -> shows restated values
key = (F.col("NPI_NUMBER") == "NPI_001") & (F.col("CUTOFF_DATE_TS") == F.lit("2026-03-31").cast("date"))
cmp = None
for d, v in refresh_versions.items():
    one = (spark.read.option("versionAsOf", v).table(q(FT_NAME)).filter(key)
             .select(F.lit(d).alias("refresh_date"), F.lit(v).alias("version"), "TOTAL_RX_CLAIMS_IN_LAST_1_MONTH"))
    cmp = one if cmp is None else cmp.unionByName(one)
display(cmp.orderBy("version"))

### Rollback (if a monthly run is bad)

If a refresh produces wrong values, you can roll back in one statement (not executed here):
```sql
RESTORE TABLE <feature_table> TO VERSION AS OF <version_of_previous_refresh>;
```
`RESTORE` itself creates a new version, so history is never lost.

## Step 5: Point-in-time lookup (the main feature-table benefit)

We ask for features at arbitrary dates. For each (NPI, date) the feature table returns the **latest cohort row with `CUTOFF_DATE_TS <= date`** — never a future row. A date before the first cohort returns NULL.

In [0]:
labels = spark.sql("""
  SELECT * FROM VALUES
    ('NPI_001', DATE'2026-01-15', 1),   -- before first cohort (2026-01-31) -> NULL features
    ('NPI_001', DATE'2026-03-15', 0),   -- -> uses cohort 2026-02-28
    ('NPI_002', DATE'2026-06-30', 1),   -- exact match -> cohort 2026-06-30
    ('NPI_003', DATE'2026-12-01', 0)    -- after last cohort -> uses 2026-10-31
  AS t(NPI_NUMBER, CUTOFF_DATE_TS, label)
""")

ts = fe.create_training_set(
    df=labels,
    feature_lookups=[FeatureLookup(
        table_name=FT_NAME,
        lookup_key="NPI_NUMBER",
        timestamp_lookup_key="CUTOFF_DATE_TS",
        feature_names=["TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_KERENDIA_IN_LAST_1_MONTH"],
    )],
    label="label",
)
pit = ts.load_df()

# Show which cohort row was matched (for transparency)
ft_rows = spark.table(q(FT_NAME)).select("NPI_NUMBER", F.col("CUTOFF_DATE_TS").alias("matched_cutoff"),
                                         F.col("TOTAL_RX_CLAIMS_IN_LAST_1_MONTH").alias("_v"))
matched = (labels.join(ft_rows, "NPI_NUMBER").filter(F.col("matched_cutoff") <= F.col("CUTOFF_DATE_TS"))
                 .groupBy("NPI_NUMBER", "CUTOFF_DATE_TS").agg(F.max("matched_cutoff").alias("expected_cutoff")))
display(pit.join(matched, ["NPI_NUMBER", "CUTOFF_DATE_TS"], "left").orderBy("NPI_NUMBER", "CUTOFF_DATE_TS"))

## Step 6: Retention — keeping only ~3 past refreshes

Delta has **no "keep N versions" setting**. History is removed by age:

1. Log entries older than `delta.logRetentionDuration` are cleaned up automatically at checkpoints.
2. Data files no longer referenced by the current version and older than `delta.deletedFileRetentionDuration` are physically deleted by `VACUUM` (run by predictive optimization if enabled, or explicitly after each monthly write).

With both set to 95 days and a monthly refresh, you keep about 3 previous refreshes. Storage cost: since each refresh overwrites the full table, expect roughly **(1 + 3) × table size**.

In this demo all versions are minutes old, so `VACUUM` removes nothing — the cell below shows which versions *would* still be restorable after 95 days.

In [0]:
# Run as the last step of each monthly job (safe: it honours the 95-day retention)
spark.sql(f"VACUUM {q(FT_NAME)}")

# Which refreshes are inside the retention window "today" vs. 95 days from now?
retention_days = 95
latest_ts = hist.agg(F.max("timestamp")).first()[0]
simulated = (hist.filter(F.col("userMetadata").startswith("refresh_date="))
    .withColumn("refresh_date", F.to_date(F.split("userMetadata", "=")[1]))
    # pretend each run happened on its refresh_date to illustrate a real monthly cadence
    .withColumn("age_days_at_last_run", F.datediff(F.lit(REFRESH_DATES[-1]).cast("date"), F.col("refresh_date")))
    .withColumn("still_time_travelable", F.col("age_days_at_last_run") <= retention_days)
    .select("version", "refresh_date", "age_days_at_last_run", "still_time_travelable")
    .orderBy("version"))
display(simulated)
# Expected on a real monthly cadence: current refresh + 3 previous (Aug, Sep, Oct) kept; July (123 days old) expires.

## Step 7: Exactly the last 3 refreshes (count-based) with a snapshot table

If you need **exactly** N snapshots regardless of run timing, keep them as rows instead of Delta versions:

* Append each refresh with a `REFRESH_DATE` column.
* Delete all rows whose `REFRESH_DATE` is not among the latest N.
* `VACUUM` (with normal retention) later reclaims the storage.

The feature table (Steps 2–5) stays the "current" table used for training/scoring; this snapshot table is the audit/history copy.

In [0]:
def append_snapshot_keep_last_n(refresh_date: str, run_idx: int, keep_n: int = KEEP_N_REFRESHES):
    df = build_refresh_df(refresh_date, run_idx, include_refresh_date=True)
    # replaceWhere makes a re-run of the same month idempotent (no duplicates)
    (df.write.format("delta").mode("overwrite")
       .option("replaceWhere", f"REFRESH_DATE = DATE'{refresh_date}'")
       .saveAsTable(q(HIST_NAME)) if spark.catalog.tableExists(q(HIST_NAME))
     else df.write.format("delta").clusterBy("REFRESH_DATE").saveAsTable(q(HIST_NAME)))
    spark.sql(f"""
      DELETE FROM {q(HIST_NAME)}
      WHERE REFRESH_DATE NOT IN (
        SELECT REFRESH_DATE FROM (SELECT DISTINCT REFRESH_DATE FROM {q(HIST_NAME)})
        ORDER BY REFRESH_DATE DESC LIMIT {keep_n})
    """)

for i, d in enumerate(REFRESH_DATES, start=1):
    append_snapshot_keep_last_n(d, i)

snap = spark.table(q(HIST_NAME)).groupBy("REFRESH_DATE").count().orderBy("REFRESH_DATE")
display(snap)
n_snap = snap.count()
assert n_snap == KEEP_N_REFRESHES, f"Expected {KEEP_N_REFRESHES} refreshes, found {n_snap}"
print(f"OK: exactly {n_snap} refreshes retained:", [str(r['REFRESH_DATE']) for r in snap.collect()])

## Summary

**Should we store the monthly DataFrame as a feature table?** Yes — as a *registration layer* on top of our computed DataFrame (not Feature Views computing from raw claims).

| | Pros | Cons |
| --- | --- | --- |
| Feature table | Features UI, point-in-time joins (no leakage), model lineage, online-serving ready, Delta time travel, UC governance | Needs `databricks-feature-engineering` lib in the job; PK must be NOT NULL + unique; table owner must add PK; untested at full scale (~8M rows, 3.9k cols) |
| Plain UC table | Time travel + governance, zero extra setup | No Features UI / PIT joins / lineage |
| Parquet | Simple file | No time travel, no PK, no UC-table lineage, manual versioning |

**Time travel:** yes — via Delta. Retention is **time-based**:
* Set `delta.logRetentionDuration` = `delta.deletedFileRetentionDuration` = `interval 95 days` → ≈ last 3 monthly refreshes.
* Ensure `VACUUM` runs (predictive optimization or last step of the job), otherwise old files are never deleted.
* For **exactly** 3 refreshes, use the snapshot pattern in Step 7.

**Monthly job template:** (1) compute DataFrame → (2) overwrite feature table with `userMetadata=refresh_date` → (3) optional snapshot append + keep last 3 → (4) `VACUUM`.

In [0]:
# Uncomment to remove the demo tables when done
# spark.sql(f"DROP TABLE IF EXISTS {q(FT_NAME)}")
# spark.sql(f"DROP TABLE IF EXISTS {q(HIST_NAME)}")